# Entity Linking (Wikidata)

Extracts named entities from a text column using `dslim/bert-base-NER` via the HuggingFace Inference API, then links the top entity per row to Wikidata via SPARQL. Adds `Entity_QID#link` and `Entity_Label` columns.

No API key required for Wikidata. A free HuggingFace token improves NER rate limits. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: use Drive (automatic) or enter credentials below
    SUAVE_TOKEN = ''   # @param {type:"string"}
    SUAVE_HOST  = ''   # @param {type:"string"}
    _p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import time

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load data and select text column

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

col_selector = widgets.Dropdown(
    options=text_cols, description='Text column:',
    layout=widgets.Layout(width='60%')
)
entity_types = widgets.SelectMultiple(
    options=['PER', 'ORG', 'LOC', 'MISC'],
    value=['PER', 'ORG', 'LOC'],
    description='Entity types:', rows=4
)
display(col_selector, entity_types)

## 2. Extract entities and link to Wikidata

In [ ]:
NER_MODEL     = 'dslim/bert-base-NER'
WIKIDATA_URL  = 'https://www.wikidata.org/w/api.php'
allowed_types = set(entity_types.value)
col           = col_selector.value

from tqdm.auto import tqdm

_wd_cache = {}

def _wikidata_lookup(entity_text):
    if entity_text in _wd_cache:
        return _wd_cache[entity_text]
    try:
        time.sleep(0.2)  # Wikidata rate limit
        resp = _req.get(WIKIDATA_URL, params={
            'action': 'wbsearchentities', 'search': entity_text,
            'language': 'en', 'format': 'json', 'limit': 1
        }, timeout=10)
        results = resp.json().get('search', [])
        if results:
            qid   = results[0]['id']
            label = results[0].get('label', entity_text)
            url   = f"https://www.wikidata.org/wiki/{qid}"
            _wd_cache[entity_text] = (url, label)
            return url, label
    except Exception:
        pass
    _wd_cache[entity_text] = (None, None)
    return None, None

qids, labels = [], []

for text in tqdm(df[col], desc='Entity linking'):
    if not text or pd.isna(text) or str(text).strip() == '':
        qids.append(None); labels.append(None); continue
    try:
        entities = client.token_classification(str(text)[:512], model=NER_MODEL)
        # Pick the highest-score entity of an allowed type
        candidates = [
            e for e in entities
            if any(t in (e.entity_group or '') for t in allowed_types)
        ]
        if not candidates:
            qids.append(None); labels.append(None); continue
        top = max(candidates, key=lambda e: e.score)
        qid, label = _wikidata_lookup(top.word)
        qids.append(qid); labels.append(label)
    except Exception:
        qids.append(None); labels.append(None)

df['Entity_QID#link'] = qids
df['Entity_Label']    = labels

n_ok = sum(1 for q in qids if q)
printmd(f"**Linked {n_ok}/{len(df)} rows to Wikidata entities.**")
display(df[[col, 'Entity_Label', 'Entity_QID#link']].dropna().head(10))

## 3. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)